# 09 — Qutrit Spectroscopy & Calibration

## Purpose

Locate the |e⟩→|f⟩ transition frequency and calibrate the e→f π-pulse amplitude, extending the qubit's operating space from a two-level (qubit) to a three-level (qutrit) system.

## Methodology

1. **e→f spectroscopy** — prepare the qubit in |e⟩ with a g→e π-pulse, then sweep the drive frequency across the expected |e⟩→|f⟩ transition. Fit a Lorentzian to extract the e→f resonance frequency.
2. **Power Rabi (e→f)** — with the qubit in |e⟩, sweep the drive gain on the `ef_ref_r180` operation and fit the Rabi oscillation to extract the π-pulse amplitude.
3. **Calibration commit** — apply the measured e→f frequency and π-pulse amplitude as calibration patches.

## Expected Outcomes

- e→f transition frequency consistent with the expected anharmonicity (typically −150 to −250 MHz for transmons)
- Clean sinusoidal Rabi oscillation for the e→f drive
- Calibrated `ef_x180` and `ef_x90` pulse amplitudes ready for qutrit experiments

## Prerequisites

- **Notebook 05** — qubit g→e frequency and π-pulse calibrated
- **Notebook 08** — e→f DRAG-Gaussian waveforms defined and registered

## 1. Setup — Session Bootstrap

Open the notebook stage and load the prior pulse waveform definition checkpoint.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    QubitSpectroscopyEF,
    PowerRabi,
    fit_quality_gate,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="09_qutrit_spectroscopy_calibration",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

pulse_waveform_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="08_pulse_waveform_definition",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"Closed a live in-memory session before reopen: {stage.had_live_session}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
print(f"Current qubit frequency: {float(attr.qb_fq) / 1e9:.6f} GHz")
print(f"Current anharmonicity: {float(attr.anharmonicity) / 1e6:.3f} MHz")
if pulse_waveform_checkpoint is not None:
    print(
        "Prior stage 08 status: "
        f"{pulse_waveform_checkpoint['status']}"
        f" ({pulse_waveform_checkpoint['summary']})"
    )

## 2. Configuration — Qutrit Spectroscopy Defaults

Define frequency sweep ranges, averaging counts, gain limits, and calibration-apply flags for both e→f spectroscopy and power Rabi measurements.

In [ ]:
APPLY_EF_SPECTROSCOPY_CALIBRATION = True
APPLY_EF_POWER_RABI_CALIBRATION = True

# --- e→f Spectroscopy defaults (Exp 15) ---
EF_SPEC_FREQ_START = 5800 * u.MHz
EF_SPEC_FREQ_END = 6000 * u.MHz
EF_SPEC_DF = 2000 * u.kHz
EF_SPEC_QB_GAIN = 1
EF_SPEC_PULSE = "x180"  # state prep: drive g→e first
EF_SPEC_N_AVG = 2000

# --- Power Rabi (e→f) defaults (Exp 12) ---
EF_RABI_MAX_GAIN = 1.2
EF_RABI_DG = 0.04
EF_RABI_OP = "ef_ref_r180"
EF_RABI_N_AVG = 10000

print("Qutrit spectroscopy & calibration settings:")
print(f"  apply e→f spectroscopy calibration: {APPLY_EF_SPECTROSCOPY_CALIBRATION}")
print(f"  apply e→f power Rabi calibration: {APPLY_EF_POWER_RABI_CALIBRATION}")
print(f"  ef spec: {EF_SPEC_FREQ_START / 1e9:.3f} – {EF_SPEC_FREQ_END / 1e9:.3f} GHz, df={EF_SPEC_DF / 1e3:.0f} kHz, n_avg={EF_SPEC_N_AVG}")
print(f"  ef Rabi: max_gain={EF_RABI_MAX_GAIN}, dg={EF_RABI_DG}, op={EF_RABI_OP}, n_avg={EF_RABI_N_AVG}")

## 3. Execution — e→f Spectroscopy

Prepare the qubit in |e⟩ via a g→e π-pulse, then sweep the drive frequency across the expected |e⟩→|f⟩ transition region. Fit the response to extract the e→f resonance frequency and apply calibration if the fit quality gate passes.

In [ ]:
ef_spec = QubitSpectroscopyEF(session)
ef_spec_result = ef_spec.run(
    pulse=EF_SPEC_PULSE,
    freq_start=EF_SPEC_FREQ_START,
    freq_end=EF_SPEC_FREQ_END,
    df=EF_SPEC_DF,
    qb_gain=EF_SPEC_QB_GAIN,
    n_avg=EF_SPEC_N_AVG,
)
ef_spec_analysis = ef_spec.analyze(ef_spec_result, update_calibration=True)
ef_spec.plot(ef_spec_analysis)

ef_spec_fit_ok = fit_quality_gate(ef_spec_analysis, r_squared_min=0.85)

if ef_spec_fit_ok:
    ef_spec_patch, ef_spec_patch_preview, ef_spec_apply_result = preview_or_apply_patch_ops(
        session,
        reason="e→f spectroscopy frequency calibration",
        proposed_patch_ops=ef_spec_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_EF_SPECTROSCOPY_CALIBRATION,
    )
    if ef_spec_apply_result is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
        if attr is None:
            raise RuntimeError("Calibration applied, but the refreshed cQED attribute snapshot is unavailable.")
else:
    print("WARNING: e→f spectroscopy fit quality gate FAILED — skipping calibration apply.")

ef_freq_hz = float(ef_spec_analysis.metrics.get("f0", np.nan))
print(f"Fitted e→f frequency: {ef_freq_hz / 1e9:.6f} GHz")
print(f"Measured anharmonicity: {(ef_freq_hz - float(attr.qb_fq)) / 1e6:.3f} MHz")

## 4. Execution — Power Rabi (e→f)

Calibrate the e→f π-pulse amplitude by sweeping gain on the `ef_ref_r180` operation with the qubit prepared in |e⟩. The Rabi oscillation is fitted to extract the optimal π and π/2 amplitudes.

In [ ]:
ef_rabi = PowerRabi(session)
ef_rabi_result = ef_rabi.run(
    max_gain=EF_RABI_MAX_GAIN,
    dg=EF_RABI_DG,
    op=EF_RABI_OP,
    n_avg=EF_RABI_N_AVG,
    transition="ef",
)
ef_rabi_analysis = ef_rabi.analyze(ef_rabi_result, update_calibration=True)
ef_rabi.plot(ef_rabi_analysis)

ef_rabi_fit_ok = fit_quality_gate(ef_rabi_analysis, r_squared_min=0.85)

if ef_rabi_fit_ok:
    ef_rabi_patch, ef_rabi_patch_preview, ef_rabi_apply_result = preview_or_apply_patch_ops(
        session,
        reason="e→f power Rabi π-pulse amplitude calibration",
        proposed_patch_ops=ef_rabi_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_EF_POWER_RABI_CALIBRATION,
    )
    if ef_rabi_apply_result is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
        if attr is None:
            raise RuntimeError("Calibration applied, but the refreshed cQED attribute snapshot is unavailable.")
else:
    print("WARNING: e→f power Rabi fit quality gate FAILED — skipping calibration apply.")

ef_a_pi = float(ef_rabi_analysis.metrics.get("a_pi", np.nan))
ef_a_pi_2 = float(ef_rabi_analysis.metrics.get("a_pi_2", np.nan))
print(f"Fitted e→f a_pi: {ef_a_pi:.4f}")
print(f"Fitted e→f a_pi/2: {ef_a_pi_2:.4f}")

## 5. Checkpoint — Save Qutrit Calibration

Record the e→f frequency and π-pulse amplitude calibration. Downstream notebooks (sideband transitions, etc.) depend on these values.

In [ ]:
ef_spec_applied = "ef_spec_apply_result" in dir() and ef_spec_apply_result is not None
ef_rabi_applied = "ef_rabi_apply_result" in dir() and ef_rabi_apply_result is not None

stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="09_qutrit_spectroscopy_calibration",
    status="calibrated" if (ef_spec_applied and ef_rabi_applied) else "characterized",
    summary="Measured e→f transition frequency and calibrated ef π-pulse amplitude.",
    consumed_inputs={
        "ef_spec_freq_start_hz": EF_SPEC_FREQ_START,
        "ef_spec_freq_end_hz": EF_SPEC_FREQ_END,
        "ef_spec_df_hz": EF_SPEC_DF,
        "ef_spec_n_avg": EF_SPEC_N_AVG,
        "ef_rabi_max_gain": EF_RABI_MAX_GAIN,
        "ef_rabi_dg": EF_RABI_DG,
        "ef_rabi_n_avg": EF_RABI_N_AVG,
    },
    persisted_outputs={
        "ef_spec_applied": ef_spec_applied,
        "ef_rabi_applied": ef_rabi_applied,
    },
    advisory_outputs={
        "ef_freq_hz": ef_freq_hz,
        "ef_a_pi": ef_a_pi,
        "ef_a_pi_2": ef_a_pi_2,
        "measured_anharmonicity_hz": ef_freq_hz - float(attr.qb_fq),
    },
    next_stage="10_sideband_transitions",
    notes=[
        "Stage 10 assumes e→f transition frequency and ef rotations are calibrated.",
        "The measured anharmonicity can be cross-checked against hardware definition.",
    ],
    metrics=dict(ef_spec_analysis.metrics) | dict(ef_rabi_analysis.metrics),
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")